In [ ]:
import pandas as pd
import numpy as np

# 1. قراءة البيانات (تأكد من تعديل اسم الملف إذا كان مختلفاً)
df = pd.read_csv('/content/kc_house_data.csv')

# 2. عرض أول 5 أسطر ومعلومات عن الأعمدة والقيم المفقودة
print("--- معاينة البيانات ---")
display(df.head())

print("\n--- معلومات الأنواع والقيم المفقودة ---")
print(df.info())

print("\n--- عدد القيم المفقودة في كل عمود ---")
print(df.isnull().sum())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# تنظيف أسماء الأعمدة من أي مسافات زائدة (سبب شائع لخطأ "column not found")
df.columns = df.columns.str.strip()

# 1. تحديد هدف التنبؤ (y) والميزات (X)
# استبدل 'price' باسم عمود سعر المنزل في ملفك إذا كان مختلفاً
target_column = 'price'

# مطابقة اسم العمود بدون حساسية لحالة الأحرف، تحسبًا لاختلاف بسيط في التسمية (Price / PRICE ...)
if target_column not in df.columns:
    match = [c for c in df.columns if c.strip().lower() == target_column.lower()]
    if match:
        target_column = match[0]
    else:
        raise KeyError(
            f"العمود '{target_column}' غير موجود في البيانات. "
            f"الأعمدة المتاحة فعليًا هي: {df.columns.tolist()}"
        )

# نحذف عمود 'id' لأنه مجرد رقم تعريف لا يفيد في التنبؤ
# ونحذف عمود 'date' بصيغته النصية الأصلية لأنه كان يسبب الخطأ التالي عند التدريب:
# ValueError: could not convert string to float: '20140527T000000'
# (بدلاً من ذلك نستخرج منه السنة والشهر كأعمدة رقمية مفيدة)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S', errors='coerce')
    df['sale_year'] = df['date'].dt.year
    df['sale_month'] = df['date'].dt.month
    df = df.drop(columns=['date'])

drop_cols = [c for c in ['id'] if c in df.columns]
df = df.drop(columns=drop_cols)

X = df.drop(columns=[target_column])
y = df[target_column]

# 2. الفصل بين الأعمدة الرقمية والنصية تلقائياً
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

# 3. بناء خطة المعالجة للأرقام: تعويض القيم المفقودة بالوسيط + تحجيم
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 4. بناء خطة المعالجة للنصوص: تعويض القيم المفقودة + تحويل إلى OneHot
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 5. دمج المعالجات في مجمع واحد
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


In [ ]:
# تقسيم البيانات إلى 80% تدريب و 20% اختبار
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"حجم بيانات التدريب: {X_train.shape[0]} عينة")
print(f"حجم بيانات الاختبار: {X_test.shape[0]} عينة")


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# نجمع خطوات المعالجة (preprocessor) مع النموذج في Pipeline واحد
# حتى يتم تجهيز البيانات (تحويل النصوص، تحجيم الأرقام) قبل التدريب مباشرة
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model.fit(X_train, y_train)

print("Model trained ✅")


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ملاحظة: هذه مسألة انحدار (توقع سعر رقمي) وليست تصنيف،
# لذلك نستخدم مقاييس الانحدار المناسبة بدل الدقة/الاستدعاء (accuracy/recall...)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE (متوسط الخطأ المطلق):", round(mae, 2))
print("MSE (متوسط مربع الخطأ):", round(mse, 2))
print("RMSE (جذر متوسط مربع الخطأ):", round(rmse, 2))
print("R2 Score:", round(r2, 4))


In [ ]:
results_df = pd.DataFrame({
    'السعر الفعلي (Actual)': y_test.values[:5],
    'السعر المتوقع (Predicted)': y_pred[:5],
    'الفرق (Difference)': np.abs(y_test.values[:5] - y_pred[:5])
})

display(results_df)
